# CS336 TinyStories training in Colab

Train a decoder-only Transformer from random weights, measure how well it predicts held-out text, and use your saved model to finish new story prompts. This is the Colab track of the project: Stanford CS336 Assignment 1 on TinyStories, run end to end on a T4. The H100 track trains a 124M Llama on FineWeb-Edu with the code in `llama_lab/`.

[**Open in Colab**](https://colab.research.google.com/github/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/CS336_Colab/CS336_TinyStories_Training.ipynb) · In Colab, select **Runtime → Change runtime type → T4 GPU**. Google Drive holds the prepared data, checkpoints, and exports, so separate sessions chain together. The raw training split is about 2.23 GB. Allow several GB of free space on Drive and in the runtime.

**Plan.** First sweep the peak learning rate with short runs, then train the full profile at the best rate.

| Step | `MODE` | Profile | Tokens processed | Iterations at batch 32 × context 256 | Estimated T4 time |
|---|---|---|---:|---:|---:|
| 1 · Learning-rate sweep | `"sweep"` | `40m` × 3 rates | 40,960,000 each | 5,000 each | about 40–90 min in total |
| 2 · Full run | `"train"` | `full` | 327,680,000 | 40,000 | about 2–4 h |
| 3 · Stories and packaging | `"generate"` or after training | — | — | — | minutes |

The T4 times are estimates from the model's arithmetic and memory traffic. The training cell prints the real throughput and remaining time every 50 iterations. Add 15–25 minutes to the first session to download the data, train the tokenizer, and encode both splits. Later sessions reuse the token files on Drive.

Both profiles use the Assignment 1 TinyStories architecture and train on randomly sampled windows from the full training split. Each run has its own learning-rate schedule sized to its budget. The handout gives **1.45** as a validation-loss target for the tuned full run and **2.00** as a low-resource reference. Your results depend on the tokenizer, evaluation sample, and training settings.

This self-study walkthrough uses Hugging Face's optimized byte-level BPE implementation to train a new tokenizer on the corpus. The model, cross-entropy, and AdamW are defined in the notebook. PyTorch supplies tensor operations, autograd, and mixed precision.

[CS336 Assignment 1 handout](https://github.com/stanford-cs336/assignment1-basics/blob/main/cs336_assignment1_basics.pdf) · [TinyStories dataset](https://huggingface.co/datasets/roneneldan/TinyStories) · [PyTorch mixed precision](https://docs.pytorch.org/docs/stable/notes/amp_examples.html) · [Folder README](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/tree/main/CS336_Colab)

## 1. Runtime and settings

Choose what this session does with `MODE`:

- `"sweep"` trains the `40m` profile once for each rate in `SWEEP_LRS`, then compares their validation losses. Finished rates are skipped, so rerunning after a disconnect continues where it stopped.
- `"train"` trains or resumes one run of `PROFILE` at `PEAK_LR`. Set `PEAK_LR` to the sweep's best rate before the full run.
- `"generate"` opens an existing model without downloading or tokenizing the dataset. `GENERATE_FROM = "github"` downloads the model committed to the repo's `CS336_Colab` folder instead of reading it from Drive.

Each run is stored under `runs/<profile>-<label>`. A blank `RUN_LABEL` becomes the learning rate, for example `full-lr1e-03`. Rerunning with the same profile and label resumes that run; changing a saved training setting under the same label is refused. For a short first check, set `MAX_UPDATES_THIS_SESSION = 100` to pause at a checkpoint and export the current model. Reset it to `None` to finish.

`FAST_ATTENTION = True` replaces the hand-written attention with PyTorch's fused kernel during training. Section 3 checks that both give the same output. The switch isn't part of the saved settings, so a run can change it between sessions.

In [ ]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    installed_tokenizers = version("tokenizers")
except PackageNotFoundError:
    installed_tokenizers = None

if installed_tokenizers != "0.22.2":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tokenizers==0.22.2"])

In [ ]:
import hashlib
import json
import math
import pickle
import shutil
import tempfile
import time
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from itertools import islice
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm

MODE = "sweep"                     # "sweep", "train" or "generate"
PROFILE = "full"                   # MODE="train" and "generate": "40m" or "full"
PEAK_LR = 1e-3                     # MODE="train": set to the sweep's best rate
SWEEP_LRS = (3e-4, 1e-3, 3e-3)     # MODE="sweep": one 40m run per rate
RUN_LABEL = ""                     # Blank: named after the learning rate
MAX_UPDATES_THIS_SESSION = None    # e.g. 100 for a short first check
MICRO_BATCH_SIZE = 32
EFFECTIVE_BATCH_SIZE = 32
FAST_ATTENTION = False             # PyTorch's fused attention for training
USE_FP16 = True
SEED = 336
LOG_EVERY = 50
EVAL_EVERY = 250
SAVE_EVERY = None                  # None: 250 for 40m, 1,000 for full
VAL_WINDOWS = 512                  # Fixed held-out windows for each check
GENERATE_FROM = "drive"            # MODE="generate": "drive" or "github"

USE_DRIVE = True
DRIVE_ROOT = Path("/content/drive/MyDrive/CS336_TinyStories")
LOCAL_ROOT = Path("/content/cs336_tinystories")
GITHUB_MODEL_URL = ("https://raw.githubusercontent.com/xqzr2000/Stanford-CS336-Llama-Adaptation/"
                    "main/CS336_Colab/model.pt")

CONTEXT_LENGTH = 256
VOCAB_SIZE = 10_000
TARGET_TOKENS = {"40m": 40_960_000, "full": 327_680_000}
TOKENS_PER_UPDATE = EFFECTIVE_BATCH_SIZE * CONTEXT_LENGTH
ACCUM_STEPS = EFFECTIVE_BATCH_SIZE // MICRO_BATCH_SIZE

if MODE not in {"sweep", "train", "generate"} or PROFILE not in TARGET_TOKENS:
    raise ValueError("Choose a valid MODE and PROFILE.")
if GENERATE_FROM not in {"drive", "github"}:
    raise ValueError('GENERATE_FROM must be "drive" or "github".')
if RUN_LABEL and not all(c.isalnum() or c in "-_" for c in RUN_LABEL):
    raise ValueError("Use letters, numbers, hyphens, or underscores for RUN_LABEL.")
if not 0 < MICRO_BATCH_SIZE <= EFFECTIVE_BATCH_SIZE or EFFECTIVE_BATCH_SIZE % MICRO_BATCH_SIZE:
    raise ValueError("MICRO_BATCH_SIZE must divide EFFECTIVE_BATCH_SIZE.")
if min(LOG_EVERY, EVAL_EVERY, SAVE_EVERY or 1, VAL_WINDOWS) < 1 or min(PEAK_LR, *SWEEP_LRS) <= 0:
    raise ValueError("Use positive intervals and learning rates.")
if MAX_UPDATES_THIS_SESSION is not None and MAX_UPDATES_THIS_SESSION < 1:
    raise ValueError("The session update limit must be positive.")
for name, tokens in TARGET_TOKENS.items():
    if tokens % TOKENS_PER_UPDATE:
        raise ValueError(f"The effective batch must divide the {name} token budget exactly.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAINING = MODE in {"sweep", "train"}
if TRAINING and device.type != "cuda":
    raise RuntimeError("Select a GPU runtime in Colab before training.")
AMP = USE_FP16 and device.type == "cuda"

def lr_label(lr):
    return f"lr{lr:.0e}"

def run_directory(profile, peak_lr, label=""):
    return root / "runs" / f"{profile}-{label or lr_label(peak_lr)}"

if USE_DRIVE and not (MODE == "generate" and GENERATE_FROM == "github"):
    from google.colab import drive
    drive.mount("/content/drive")
    root = DRIVE_ROOT
else:
    root = LOCAL_ROOT
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
(root / "runs").mkdir(parents=True, exist_ok=True)

print(torch.cuda.get_device_name() if device.type == "cuda" else "CPU")
print(f"Precision: {'FP16 autocast with gradient scaling' if AMP else 'FP32'} | "
      f"attention: {'fused' if FAST_ATTENTION else 'hand-written'}")
if MODE == "sweep":
    print(f"Sweep: 40m × {len(SWEEP_LRS)} rates {[lr_label(lr) for lr in SWEEP_LRS]}")
elif MODE == "generate" and GENERATE_FROM == "github":
    print(f"generate: model.pt from {GITHUB_MODEL_URL}")
else:
    print(f"{MODE}: {run_directory(PROFILE, PEAK_LR, RUN_LABEL)}")

## 2. Train the tokenizer and prepare token IDs

The course uses the GPT-4 version of TinyStories. The two raw text files mark story boundaries with `<|endoftext|>`. Byte-level BPE begins with a byte alphabet and learns merges of frequently adjacent symbols. We train the merges on the training split, then use that same tokenizer on the validation split. Each story contributes its text IDs and one end-of-story ID.

The first run downloads and encodes both splits. Later sessions reuse the token files on Drive and copy them to the Colab VM for fast random reads. Each token ID fits in an unsigned 16-bit integer, so the token files take two bytes per token.

In [ ]:
DATA_REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64"
DATA_URL = f"https://huggingface.co/datasets/roneneldan/TinyStories/resolve/{DATA_REVISION}"
DATA_NAME = "TinyStoriesV2-GPT4"
EOT = "<|endoftext|>"

def atomic_write(path, writer):
    # Publish a completed file under its final name.
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix=".part", delete=False) as f:
        partial = Path(f.name)
    try:
        writer(partial)
        partial.replace(path)
    finally:
        partial.unlink(missing_ok=True)

def stage_copy(source, destination):
    if source != destination:
        atomic_write(destination, lambda path: shutil.copyfile(source, path))

def download_split(split, destination):
    if destination.exists():
        return
    url = f"{DATA_URL}/{DATA_NAME}-{split}.txt"

    def download(path):
        with requests.get(url, stream=True, timeout=(30, 120)) as response:
            response.raise_for_status()
            size = int(response.headers.get("Content-Length", 0))
            with path.open("wb") as output, tqdm(
                total=size or None, unit="B", unit_scale=True, desc=f"Download {split}"
            ) as progress:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    output.write(chunk)
                    progress.update(len(chunk))
    atomic_write(destination, download)

def stories(path):
    lines = []
    with path.open(encoding="utf-8") as source:
        for line in source:
            if line.strip() == EOT:
                story = "".join(lines)
                if story.strip():
                    yield story
                lines.clear()
            else:
                lines.append(line)
    story = "".join(lines)
    if story.strip():
        yield story

def story_batches(path, size=2048):
    iterator = iter(stories(path))
    while batch := list(islice(iterator, size)):
        yield batch

def train_tokenizer(train_text):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        special_tokens=[EOT],
    )
    tokenizer.train_from_iterator(story_batches(train_text), trainer=trainer)
    return tokenizer

def encode_split(tokenizer, raw_path, destination):
    eos_id = tokenizer.token_to_id(EOT)

    def write_tokens(path):
        with path.open("wb") as output:
            for batch in tqdm(story_batches(raw_path), desc=f"Encode {raw_path.stem}"):
                encodings = tokenizer.encode_batch(batch, add_special_tokens=False)
                ids = [token_id for encoding in encodings for token_id in (*encoding.ids, eos_id)]
                np.asarray(ids, dtype="<u2").tofile(output)
    atomic_write(destination, write_tokens)

In [ ]:
if TRAINING:
    data_key = f"tinystories-{DATA_REVISION[:8]}-bytebpe{VOCAB_SIZE}-v1"
    cache_dir = root / "data" / data_key
    work_dir = LOCAL_ROOT / "data" / data_key
    raw_paths = {split: work_dir / f"{split}.txt" for split in ("train", "valid")}
    tokenizer_path = cache_dir / "tokenizer.json"

    if tokenizer_path.exists():
        tokenizer_json = tokenizer_path.read_text(encoding="utf-8")
        tokenizer = Tokenizer.from_str(tokenizer_json)
    else:
        download_split("train", raw_paths["train"])
        tokenizer = train_tokenizer(raw_paths["train"])
        tokenizer_json = tokenizer.to_str()
        atomic_write(tokenizer_path, lambda path: path.write_text(tokenizer_json, encoding="utf-8"))

    tokenizer_hash = hashlib.sha256(tokenizer_json.encode("utf-8")).hexdigest()
    if tokenizer.get_vocab_size() != VOCAB_SIZE or tokenizer.token_to_id(EOT) is None:
        raise ValueError("The cached tokenizer has the wrong vocabulary or is missing the story marker.")

    streams = {}
    for split, raw_path in raw_paths.items():
        name = f"{split}-{tokenizer_hash[:16]}.bin"
        cached_file = cache_dir / name
        local_file = work_dir / name
        if not cached_file.exists():
            download_split(split, raw_path)
            encode_split(tokenizer, raw_path, local_file)
            stage_copy(local_file, cached_file)
        elif not local_file.exists() or local_file.stat().st_size != cached_file.stat().st_size:
            stage_copy(cached_file, local_file)
        streams[split] = np.memmap(local_file, dtype="<u2", mode="r")
        print(f"{split}: {len(streams[split]):,} token IDs")

    example = "Once upon a time, a little robot found a blue kite."
    encoded = tokenizer.encode(example)
    print(encoded.tokens)
    print(encoded.ids)
    assert tokenizer.decode(encoded.ids) == example

## 3. Build the Transformer

A batch of token IDs has shape $[B,T]$. The embedding maps it to $[B,T,D]$. Each pre-norm residual block uses RMSNorm, causal multi-head attention, and a SwiGLU feed-forward layer. The output projection produces $[B,T,V]$ logits, where $V$ is the vocabulary size.

- **RMSNorm** divides by the root mean square across features, then multiplies by a learned scale: $\operatorname{RMSNorm}(x)_i = g_i\,x_i\big/\sqrt{\tfrac{1}{D}\sum_j x_j^2+\epsilon}$.
- **RoPE** rotates the queries and keys in feature pairs by angles that depend on position. The dot product then depends on the relative rotation between positions.
- **Causal attention** scores each query against every key, $S=QK^\top/\sqrt{d_h}$. The mask removes future keys before softmax, and the output is $\operatorname{softmax}(S)\,V$ for each head.
- **SwiGLU** computes $\operatorname{SiLU}(xW_1)\odot(xW_3)$ before projecting back to model width with $W_2$.

The default model has 4 layers, width 512, 16 query/key/value heads, feed-forward width 1,344, and separate input embeddings and output weights: 22,696,448 parameters.

With `FAST_ATTENTION = True`, the attention layer calls `F.scaled_dot_product_attention(q, k, v, is_causal=True)` on the same rotated queries and keys. It computes the same equation without storing the full $T\times T$ score matrix in memory, which is the slowest part of the hand-written version on a T4. The next cells define the model and check that both paths agree.

In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    vocab_size: int = 10_000
    context_length: int = 256
    width: int = 512
    heads: int = 16
    layers: int = 4
    ff_width: int = 1344
    rope_theta: float = 10_000.0

class Linear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        std = math.sqrt(2 / (in_features + out_features))
        nn.init.trunc_normal_(self.weight, std=std, a=-3 * std, b=3 * std)

    def forward(self, x):
        return x @ self.weight.T

class RMSNorm(nn.Module):
    def __init__(self, width, eps=1e-5):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(width))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        normalized = xf * torch.rsqrt(xf.square().mean(dim=-1, keepdim=True) + self.eps)
        return (normalized * self.scale).to(x.dtype)

class RotaryEmbedding(nn.Module):
    def __init__(self, length, head_dim, theta):
        super().__init__()
        inverse_frequency = theta ** (-torch.arange(0, head_dim, 2).float() / head_dim)
        angles = torch.outer(torch.arange(length).float(), inverse_frequency)
        self.register_buffer("cos", angles.cos(), persistent=False)
        self.register_buffer("sin", angles.sin(), persistent=False)

    def forward(self, x):
        length = x.size(-2)
        cos = self.cos[:length].to(x.dtype)
        sin = self.sin[:length].to(x.dtype)
        even, odd = x[..., 0::2], x[..., 1::2]
        return torch.stack((even * cos - odd * sin, odd * cos + even * sin), dim=-1).flatten(-2)

class CausalAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = config.heads
        self.head_dim = config.width // config.heads
        self.query = Linear(config.width, config.width)
        self.key = Linear(config.width, config.width)
        self.value = Linear(config.width, config.width)
        self.output = Linear(config.width, config.width)
        self.rope = RotaryEmbedding(config.context_length, self.head_dim, config.rope_theta)
        future = torch.ones(config.context_length, config.context_length, dtype=torch.bool).triu(1)
        self.register_buffer("future", future, persistent=False)

    fast = False  # Set from FAST_ATTENTION below.

    def forward(self, x):
        batch, length, width = x.shape

        def split_heads(z):
            return z.reshape(batch, length, self.heads, self.head_dim).transpose(1, 2)

        q = self.rope(split_heads(self.query(x)))
        k = self.rope(split_heads(self.key(x)))
        v = split_heads(self.value(x))
        if self.fast:
            combined = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            return self.output(combined.transpose(1, 2).reshape(batch, length, width))
        scores = (q @ k.transpose(-1, -2)).float() / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.future[:length, :length], -torch.inf)
        attention = scores.softmax(dim=-1).to(v.dtype)
        combined = (attention @ v).transpose(1, 2).reshape(batch, length, width)
        return self.output(combined)

class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate = Linear(config.width, config.ff_width)
        self.up = Linear(config.width, config.ff_width)
        self.down = Linear(config.ff_width, config.width)

    def forward(self, x):
        gate = self.gate(x)
        return self.down((gate * gate.sigmoid()) * self.up(x))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn_norm = RMSNorm(config.width)
        self.attention = CausalAttention(config)
        self.ff_norm = RMSNorm(config.width)
        self.ff = SwiGLU(config)

    def forward(self, x):
        x = x + self.attention(self.attn_norm(x))
        return x + self.ff(self.ff_norm(x))

class TinyStoriesLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        if config.width % (2 * config.heads):
            raise ValueError("The attention head dimension must be an even integer for RoPE.")
        self.config = config
        self.embedding = nn.Parameter(torch.empty(config.vocab_size, config.width))
        nn.init.trunc_normal_(self.embedding, std=1.0, a=-3.0, b=3.0)
        self.blocks = nn.ModuleList(Block(config) for _ in range(config.layers))
        self.norm = RMSNorm(config.width)
        self.lm_head = Linear(config.width, config.vocab_size)

    def forward(self, token_ids):
        if not 1 <= token_ids.size(1) <= self.config.context_length:
            raise ValueError("Sequence length is outside the model context.")
        x = self.embedding[token_ids]
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.norm(x))

CausalAttention.fast = FAST_ATTENTION

In [ ]:
# Both attention paths on the same small model and input.
check_config = ModelConfig(vocab_size=64, context_length=32, width=64, heads=4, layers=2, ff_width=128)
torch.manual_seed(SEED)
check_model = TinyStoriesLM(check_config).to(device).eval()
check_ids = torch.randint(0, 64, (2, 32), device=device)
outputs = {}
with torch.inference_mode():
    for fast in (False, True):
        CausalAttention.fast = fast
        outputs[fast] = check_model(check_ids).float()
CausalAttention.fast = FAST_ATTENTION
torch.testing.assert_close(outputs[True], outputs[False], atol=1e-4, rtol=1e-4)
print(f"Fused and hand-written attention agree in FP32 on {device}.")
if device.type == "cuda":
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        CausalAttention.fast = False
        slow = check_model(check_ids).float()
        CausalAttention.fast = True
        fast = check_model(check_ids).float()
    CausalAttention.fast = FAST_ATTENTION
    print(f"Largest FP16 logit difference: {(fast - slow).abs().max().item():.1e}")
del check_model

## 4. Predict the next token

A window of $T+1$ IDs gives $T$ input IDs and $T$ targets shifted one position forward. For target ID $y$ and vocabulary logits $z$, the loss is $\log\sum_j e^{z_j}-z_y$. We compute the reduction in FP32 and average over all positions. Perplexity is $e^{\text{validation loss}}$.

Validation during training uses the same `VAL_WINDOWS` randomly selected windows at each check, so checkpoints are compared on identical text. When a run finishes, the notebook also scores the **entire** validation split in non-overlapping windows and stores that number as the run's final result.

The optimizer is AdamW. It keeps exponential moving averages $m$ and $v$ of the gradients and of their squares, corrects their initialization bias, and applies weight decay separately from the gradient update:

$$
\theta \leftarrow \theta(1-\eta\lambda) - \eta\,\frac{m/(1-\beta_1^t)}{\sqrt{v/(1-\beta_2^t)}+\epsilon}.
$$

The learning rate warms up linearly over the first 5% of iterations, then follows a cosine down to 10% of the peak:

$$
\eta(s)=\begin{cases}\eta_{\max}\,s/W & s\le W\\[4pt] \eta_{\max}\big[0.1+0.45\,(1+\cos(\pi\,\tfrac{s-W}{S-W}))\big] & s>W.\end{cases}
$$

Run the following cell to define batching, loss, AdamW, the schedule, and validation.

In [ ]:
def next_token_batch(stream, batch_size, context, generator):
    if len(stream) <= context:
        raise ValueError("The token stream is shorter than one training window.")
    starts = torch.randint(len(stream) - context, (batch_size,), generator=generator).numpy()
    indices = starts[:, None] + np.arange(context + 1)
    windows = torch.from_numpy(stream[indices].astype(np.int64)).to(device)
    return windows[:, :-1], windows[:, 1:]

def next_token_loss(logits, targets):
    logits = logits.float()
    selected = logits.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    return (torch.logsumexp(logits, dim=-1) - selected).mean()

class AdamW(torch.optim.Optimizer):
    def __init__(self, parameters, lr, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1):
        super().__init__(parameters, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr, (beta1, beta2) = group["lr"], group["betas"]
            for parameter in group["params"]:
                grad = parameter.grad
                if grad is None:
                    continue
                state = self.state[parameter]
                if not state:
                    state["step"] = 0
                    state["mean"] = torch.zeros_like(parameter)
                    state["square_mean"] = torch.zeros_like(parameter)
                state["step"] += 1
                step = state["step"]
                mean, square_mean = state["mean"], state["square_mean"]
                mean.lerp_(grad, 1 - beta1)
                square_mean.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                denominator = square_mean.sqrt().div_(math.sqrt(1 - beta2 ** step))
                denominator.add_(group["eps"])
                parameter.mul_(1 - lr * group["weight_decay"])
                parameter.addcdiv_(mean, denominator, value=-lr / (1 - beta1 ** step))

def learning_rate(step, total_steps, peak_lr):
    warmup = max(1, int(total_steps * 0.05))
    if step <= warmup:
        return peak_lr * step / warmup
    progress = (step - warmup) / (total_steps - warmup)
    return peak_lr * (0.1 + 0.45 * (1 + math.cos(math.pi * progress)))

def mixed_precision():
    return torch.autocast("cuda", dtype=torch.float16) if AMP else nullcontext()

@torch.inference_mode()
def validation_loss(model, stream):
    """Mean loss on VAL_WINDOWS fixed random windows."""
    model.eval()
    context = model.config.context_length
    generator = torch.Generator().manual_seed(SEED + 2)
    # Draw every start at once so the windows don't depend on MICRO_BATCH_SIZE.
    all_starts = torch.randint(len(stream) - context, (VAL_WINDOWS,), generator=generator).numpy()
    losses = []
    for first in range(0, VAL_WINDOWS, MICRO_BATCH_SIZE):
        starts = all_starts[first:first + MICRO_BATCH_SIZE]
        ids = torch.from_numpy(stream[starts[:, None] + np.arange(context + 1)].astype(np.int64)).to(device)
        with mixed_precision():
            losses.append(next_token_loss(model(ids[:, :-1]), ids[:, 1:]) * len(starts))
    model.train()
    return (torch.stack(losses).sum() / VAL_WINDOWS).item()

@torch.inference_mode()
def full_validation_loss(model, stream, batch_size=64):
    """Mean loss over the whole validation split in non-overlapping windows."""
    model.eval()
    context = model.config.context_length
    windows = (len(stream) - 1) // context
    total = torch.zeros((), device=device, dtype=torch.float64)
    for first in range(0, windows, batch_size):
        starts = np.arange(first, min(first + batch_size, windows)) * context
        indices = starts[:, None] + np.arange(context + 1)
        ids = torch.from_numpy(stream[indices].astype(np.int64)).to(device)
        with mixed_precision():
            total += next_token_loss(model(ids[:, :-1]), ids[:, 1:]).double() * len(starts)
    model.train()
    return (total / windows).item(), windows * context

## 5. Save and resume

`checkpoint-*.pt` stores weights, optimizer moments, the FP16 gradient scaler, the batch sampler's random state, and the training history, about 272 MB at full size. The notebook keeps the two latest checkpoints. By default it saves every 250 iterations for `40m` and every 1,000 for `full`. Writing to Drive takes several seconds, so saving every 250 iterations of the full run would cost roughly 10% of its time; at 1,000, a disconnect loses a few minutes of training. If the runtime disconnects, reopen the notebook, keep the same `MODE`, profile, rate and label, then rerun the cells to resume from the last saved iteration.

`model.pt` is the inference export. It packages the weights in FP16, the architecture, the tokenizer, and the validation results together, about 46 MB, small enough to commit to GitHub. Loading converts the weights back to FP32. Each planned pause or completed run refreshes this file. `metrics.json` records the run's settings and its training and validation history.

In [ ]:
def save_torch(payload, destination):
    # Serialize on the VM disk first, then copy to the destination.
    with tempfile.TemporaryDirectory(dir=LOCAL_ROOT) as directory:
        local_file = Path(directory) / destination.name
        torch.save(payload, local_file)
        stage_copy(local_file, destination)

def write_metrics(run_dir, spec, history, **extra):
    payload = {"run": run_dir.name, "spec": spec, "history": history, **extra}
    atomic_write(run_dir / "metrics.json",
                 lambda path: path.write_text(json.dumps(payload, indent=1), encoding="utf-8"))

def save_checkpoint(run_dir, model, optimizer, scaler, sampler, spec, step, history):
    destination = run_dir / f"checkpoint-{step:08d}.pt"
    save_torch({
        "spec": spec,
        "step": step,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "sampler": sampler.get_state(),
        "history": history,
    }, destination)
    for old in sorted(run_dir.glob("checkpoint-*.pt"), reverse=True)[2:]:
        old.unlink()
    write_metrics(run_dir, spec, history)
    print(f"Saved {destination.name}")

def resume_training(run_dir, model, optimizer, scaler, sampler, spec):
    checkpoints = sorted(run_dir.glob("checkpoint-*.pt"), reverse=True)
    if not checkpoints:
        return 0, []
    for checkpoint in checkpoints:
        try:
            state = torch.load(checkpoint, map_location="cpu", weights_only=True)
        except (OSError, RuntimeError, EOFError, IndexError, pickle.UnpicklingError) as error:
            print(f"Could not read {checkpoint.name}: {error}")
            continue
        if state["spec"] != spec:
            raise ValueError("Saved training settings differ. Restore those settings or use a new RUN_LABEL.")
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        sampler.set_state(state["sampler"])
        print(f"Resuming after iteration {state['step']:,}")
        return state["step"], state["history"]
    raise RuntimeError("No readable checkpoint found. Inspect the saved files before starting a new run.")

def export_model(run_dir, model, tokenizer_json, step, total_steps, loss, full_loss=None):
    destination = run_dir / "model.pt"
    save_torch({
        "format_version": 2,
        "config": asdict(model.config),
        "model": {name: value.detach().cpu().half() for name, value in model.state_dict().items()},
        "tokenizer_json": tokenizer_json,
        "step": step,
        "total_steps": total_steps,
        "validation_loss": loss,
        "full_validation_loss": full_loss,
        "run": run_dir.name,
    }, destination)
    print(f"Inference export: {destination} ({destination.stat().st_size / 1e6:.1f} MB)")

def load_export(path):
    saved = torch.load(path, map_location="cpu", weights_only=True)
    if saved["format_version"] not in (1, 2):
        raise ValueError("This model export uses a different format version.")
    tokenizer = Tokenizer.from_str(saved["tokenizer_json"])
    model = TinyStoriesLM(ModelConfig(**saved["config"]))
    model.load_state_dict({name: value.float() for name, value in saved["model"].items()})
    return model.to(device).eval(), tokenizer, saved

## 6. Train on the T4

The default batch is 32 sequences in a single pass, so each optimizer update averages the loss over 32 × 256 target positions. A smaller `MICRO_BATCH_SIZE` accumulates gradients over several passes: each micro-batch loss is divided by the number of passes, so the accumulated gradient is still an average over all 32 sequences. We keep the model and optimizer state in FP32, use FP16 autocast for eligible GPU operations, and unscale gradients before clipping them. The scaler can skip an optimizer update if the gradients overflow. The iteration and token counters still include that batch.

Use the steady-state tokens per second and the ETA printed here to plan your Colab time. The ETA covers training operations and excludes validation and checkpoint writes. A run that reaches its last iteration scores the whole validation split, about a minute on a T4, and exports `model.pt`. This cell only defines the training function; the sweep and full-run sections call it.

In [ ]:
def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize()

def train(profile, peak_lr, label=""):
    """Train or resume one run and return its directory and history."""
    run_dir = run_directory(profile, peak_lr, label)
    run_dir.mkdir(parents=True, exist_ok=True)
    total_updates = TARGET_TOKENS[profile] // TOKENS_PER_UPDATE
    save_every = SAVE_EVERY or (250 if profile == "40m" else 1_000)
    torch.manual_seed(SEED)
    sampler = torch.Generator().manual_seed(SEED + 1)
    config = ModelConfig(vocab_size=VOCAB_SIZE, context_length=CONTEXT_LENGTH)
    model = TinyStoriesLM(config).to(device)
    print(f"{run_dir.name}: {total_updates:,} iterations | "
          f"{sum(p.numel() for p in model.parameters()):,} parameters")
    optimizer = AdamW(model.parameters(), lr=peak_lr)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)

    spec = {
        "config": asdict(config),
        "data_revision": DATA_REVISION,
        "tokenizer_hash": tokenizer_hash,
        "effective_batch": EFFECTIVE_BATCH_SIZE,
        "total_steps": total_updates,
        "peak_lr": peak_lr,
        "seed": SEED,
        "fp16": AMP,
        "val_windows": VAL_WINDOWS,
    }
    start_step, history = resume_training(run_dir, model, optimizer, scaler, sampler, spec)
    final_step = min(total_updates, start_step + (MAX_UPDATES_THIS_SESSION or total_updates))
    last_val = next((row["val_loss"] for row in reversed(history) if "val_loss" in row), None)
    model.train()

    interval_steps = 0
    interval_seconds = 0.0
    interval_loss = torch.zeros((), device=device)
    block_start = time.perf_counter()

    for step in range(start_step + 1, final_step + 1):
        lr = learning_rate(step, total_updates, peak_lr)
        for group in optimizer.param_groups:
            group["lr"] = lr
        optimizer.zero_grad(set_to_none=True)
        for _ in range(ACCUM_STEPS):
            x, y = next_token_batch(streams["train"], MICRO_BATCH_SIZE, config.context_length, sampler)
            with mixed_precision():
                loss = next_token_loss(model(x), y) / ACCUM_STEPS
            scaler.scale(loss).backward()
            interval_loss += loss.detach()
        scaler.unscale_(optimizer)
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        if not AMP and not torch.isfinite(grad_norm):
            raise FloatingPointError("Gradient norm is non-finite. Lower the learning rate.")
        scaler.step(optimizer)
        scaler.update()
        interval_steps += 1

        should_log = step % LOG_EVERY == 0 or step == final_step
        should_eval = step % EVAL_EVERY == 0 or step == final_step
        should_save = step % save_every == 0 or step == final_step
        if should_log or should_eval or should_save:
            synchronize()
            interval_seconds += time.perf_counter() - block_start
        if should_log:
            mean_loss = interval_loss.item() / interval_steps
            if not math.isfinite(mean_loss):
                raise FloatingPointError("Training loss is non-finite. Check precision and learning rate.")
            rate = interval_steps * TOKENS_PER_UPDATE / interval_seconds
            eta_hours = (total_updates - step) * TOKENS_PER_UPDATE / rate / 3600
            print(f"{step:,}/{total_updates:,} | loss {mean_loss:.3f} | lr {lr:.2e} | "
                  f"{rate:,.0f} tok/s | ETA {eta_hours:.2f} h")
            history.append(dict(step=step, train_loss=mean_loss, tokens_per_second=rate))
            interval_loss.zero_()
            interval_steps, interval_seconds = 0, 0.0
        if should_eval:
            last_val = validation_loss(model, streams["valid"])
            if not math.isfinite(last_val):
                raise FloatingPointError("Validation loss is non-finite.")
            history.append(dict(step=step, val_loss=last_val))
            print(f"Validation loss {last_val:.3f} | perplexity {math.exp(last_val):.2f}")
        if should_save:
            save_checkpoint(run_dir, model, optimizer, scaler, sampler, spec, step, history)
        if should_log or should_eval or should_save:
            block_start = time.perf_counter()

    if start_step == final_step == total_updates and (run_dir / "model.pt").exists():
        print(f"{run_dir.name} already finished at {final_step:,} iterations.")
        return run_dir, history
    full_loss = None
    if final_step == total_updates:
        full_loss, scored = full_validation_loss(model, streams["valid"])
        print(f"Full validation split: loss {full_loss:.4f} | perplexity {math.exp(full_loss):.2f} | "
              f"{scored:,} target tokens")
    export_model(run_dir, model, tokenizer_json, final_step, total_updates, last_val, full_loss)
    write_metrics(run_dir, spec, history, finished=final_step == total_updates,
                  final_step=final_step, full_validation_loss=full_loss)
    return run_dir, history

## 7. Sweep the learning rate on the 40m profile

The peak learning rate matters more than anything else in this notebook for reaching the handout's targets. Each rate in `SWEEP_LRS` trains a fresh `40m` run with the same seed, data order, and schedule shape, so the only difference is $\eta_{\max}$. A rate that diverges stops with a non-finite loss and is reported as such.

Pick the rate with the lowest full-split validation loss. If the best rate is the largest one tried, add a higher rate to `SWEEP_LRS` and rerun the sweep; finished runs are skipped. The comparison is saved as `lr_sweep.json` and `lr_sweep.png` in the Drive folder.

In [ ]:
def summarize_sweep():
    rows = []
    for lr in SWEEP_LRS:
        metrics_path = run_directory("40m", lr) / "metrics.json"
        if not metrics_path.exists():
            continue
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        checks = [row for row in metrics["history"] if "val_loss" in row]
        rows.append(dict(
            peak_lr=lr, finished=metrics.get("finished", False), diverged=metrics.get("diverged", False),
            last_step=checks[-1]["step"] if checks else 0,
            val_loss=checks[-1]["val_loss"] if checks else None,
            full_validation_loss=metrics.get("full_validation_loss"), history=checks,
        ))
    return rows

if MODE == "sweep":
    for lr in SWEEP_LRS:
        previous = run_directory("40m", lr) / "metrics.json"
        if previous.exists() and json.loads(previous.read_text()).get("diverged"):
            print(f"{lr_label(lr)} diverged in an earlier session. Skipping it.")
            continue
        try:
            train("40m", lr)
        except FloatingPointError as error:
            print(f"{lr_label(lr)} diverged: {error}")
            run_dir = run_directory("40m", lr)
            metrics = json.loads((run_dir / "metrics.json").read_text()) if (run_dir / "metrics.json").exists() else {}
            write_metrics(run_dir, metrics.get("spec"), metrics.get("history", []), diverged=True, finished=False)

    sweep = summarize_sweep()
    print(f"\n{'peak LR':>9} {'status':>10} {'last step':>10} {'sampled val':>12} {'full val':>9}")
    for row in sweep:
        status = "diverged" if row["diverged"] else "finished" if row["finished"] else "partial"
        sampled = f"{row['val_loss']:.4f}" if row["val_loss"] is not None else "—"
        full = f"{row['full_validation_loss']:.4f}" if row["full_validation_loss"] is not None else "—"
        print(f"{row['peak_lr']:>9.0e} {status:>10} {row['last_step']:>10,} {sampled:>12} {full:>9}")
    finished = [row for row in sweep if row["finished"] and row["full_validation_loss"] is not None]
    if finished:
        best = min(finished, key=lambda row: row["full_validation_loss"])
        print(f"\nBest rate: {best['peak_lr']:.0e} (full validation loss {best['full_validation_loss']:.4f}). "
              f"Set PEAK_LR = {best['peak_lr']:.0e} and MODE = \"train\" for the full run.")
        if best["peak_lr"] == max(SWEEP_LRS):
            print("The best rate is the largest one tried. Consider adding a higher rate to SWEEP_LRS.")

    fig, ax = plt.subplots(figsize=(9, 4))
    for row in sweep:
        ax.plot([r["step"] * TOKENS_PER_UPDATE / 1e6 for r in row["history"]],
                [r["val_loss"] for r in row["history"]], marker="o", markersize=3,
                label=f"{row['peak_lr']:.0e}" + (" (diverged)" if row["diverged"] else ""))
    ax.axhline(2.00, color="gray", linestyle="--", linewidth=1, label="handout 40M reference (2.00)")
    ax.set(xlabel="Training tokens processed (millions)", ylabel="Validation loss", title="40m learning-rate sweep")
    ax.grid(alpha=0.25)
    ax.legend(title="peak LR")
    sweep_png = root / "runs" / "lr_sweep.png"
    atomic_write(sweep_png, lambda path: fig.savefig(path, format="png", dpi=150, bbox_inches="tight"))
    atomic_write(root / "runs" / "lr_sweep.json",
                 lambda path: path.write_text(json.dumps(sweep, indent=1), encoding="utf-8"))
    plt.show()

## 8. Train the full profile

Set `MODE = "train"`, `PROFILE = "full"`, and `PEAK_LR` to the sweep's best rate, then run the notebook from the top. The run resumes automatically after a disconnect as long as the profile, rate, and label stay the same. Expect several hours on a T4; the log prints the remaining time.

In [ ]:
if MODE == "train":
    run_dir, history = train(PROFILE, PEAK_LR, RUN_LABEL)

## 9. Plot the learning curves

Training loss averages over the recent sampled training batches. Validation loss measures the fixed held-out windows. The dashed lines mark the handout's references: 2.00 for the 40M-token run and 1.45 for the tuned full run. The plot is saved as `learning_curves.png` next to the run.

In [ ]:
if MODE == "train":
    fig, ax = plt.subplots(figsize=(9, 4))
    for key, label in [("train_loss", "Training"), ("val_loss", "Validation")]:
        records = [row for row in history if key in row]
        ax.plot(
            [row["step"] * TOKENS_PER_UPDATE / 1e6 for row in records],
            [row[key] for row in records],
            label=label,
            marker="o" if key == "val_loss" else None,
            markersize=3,
        )
    reference = 1.45 if PROFILE == "full" else 2.00
    ax.axhline(reference, color="gray", linestyle="--", linewidth=1, label=f"handout reference ({reference:.2f})")
    ax.set(xlabel="Training tokens processed (millions)", ylabel="Cross-entropy loss", title=run_dir.name)
    ax.grid(alpha=0.25)
    ax.legend()
    atomic_write(run_dir / "learning_curves.png",
                 lambda path: fig.savefig(path, format="png", dpi=150, bbox_inches="tight"))
    plt.show()

## 10. Reload and generate a story

This cell always loads an export from disk. With `MODE = "generate"`, choose where it comes from: `GENERATE_FROM = "drive"` reads `model.pt` for the selected profile, rate and label; `"github"` downloads the model committed to the repo's `CS336_Colab` folder, with no Drive or dataset needed. A CPU runtime works for generation.

At each generation step the model predicts a distribution for the next token. Temperature $\tau$ rescales the logits, $p=\operatorname{softmax}(z/\tau)$, so lower values favor likely tokens more sharply. Top-$k$ sampling restricts the choice to the $k$ highest-scoring tokens and renormalizes. Generation ends when the model emits the story delimiter or reaches the new-token limit. This implementation recomputes attention on the most recent context at each step.

In [ ]:
@torch.inference_mode()
def generate(model, tokenizer, prompt, max_new_tokens=160, temperature=0.8, top_k=40, seed=42):
    if max_new_tokens < 0 or temperature <= 0 or top_k < 1:
        raise ValueError("Use nonnegative max_new_tokens, positive temperature, and top_k >= 1.")
    eos_id = tokenizer.token_to_id(EOT)
    ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    if not ids:
        raise ValueError("Enter a non-empty story prompt.")
    rng = torch.Generator(device=device).manual_seed(seed)
    for _ in range(max_new_tokens):
        recent = torch.tensor([ids[-model.config.context_length:]], dtype=torch.long, device=device)
        logits = model(recent)[0, -1].float() / temperature
        values, choices = torch.topk(logits, min(top_k, logits.numel()))
        choice = torch.multinomial(values.softmax(dim=-1), num_samples=1, generator=rng)
        token_id = choices[choice].item()
        if token_id == eos_id:
            break
        ids.append(token_id)
    return tokenizer.decode(ids)

def fetch_github_model(destination):
    if destination.exists():
        return destination
    with requests.get(GITHUB_MODEL_URL, stream=True, timeout=(30, 120)) as response:
        if response.status_code == 404:
            raise FileNotFoundError("No model.pt in the repo's CS336_Colab folder yet. Use GENERATE_FROM = \"drive\".")
        response.raise_for_status()
        atomic_write(destination, lambda path: path.write_bytes(response.content))
    return destination

if MODE == "sweep":
    finished = [row for row in summarize_sweep() if row["finished"] and row["full_validation_loss"] is not None]
    if not finished:
        raise RuntimeError("No finished sweep run to load yet.")
    best = min(finished, key=lambda row: row["full_validation_loss"])
    run_dir = run_directory("40m", best["peak_lr"])
elif MODE == "generate" and GENERATE_FROM == "github":
    run_dir = LOCAL_ROOT / "github"
    fetch_github_model(run_dir / "model.pt")
else:
    run_dir = run_directory(PROFILE, PEAK_LR, RUN_LABEL)
MODEL_PATH = run_dir / "model.pt"
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"{MODEL_PATH} is missing. Run training or choose the saved profile, rate and label.")

story_model, story_tokenizer, saved = load_export(MODEL_PATH)
full = saved.get("full_validation_loss")
print(f"Loaded {saved.get('run', MODEL_PATH.parent.name)} at iteration {saved['step']:,} | "
      f"sampled validation loss {saved['validation_loss']:.3f}"
      + (f" | full validation loss {full:.3f}" if full is not None else "") + "\n")
PROMPT = "Once upon a time, a little robot found a blue kite."
print(generate(story_model, story_tokenizer, PROMPT))

## 11. Package the results for `CS336_Colab`

This cell writes the files that go into the repo's `CS336_Colab` folder to `CS336_Colab_upload` in your Drive folder:

| File | Contents |
| --- | --- |
| `model.pt` | FP16 weights, configuration, tokenizer, and validation results, about 46 MB |
| `metrics.json` | Run settings, training and validation history, and the full-split validation loss |
| `learning_curves.png` | The plot from section 9 |
| `samples.md` | Stories for three fixed prompts with the sampling settings and seed |
| `lr_sweep.json`, `lr_sweep.png` | The sweep comparison, if the sweep ran |

Run it after the full run finishes, in the same session or with `MODE = "generate"` and the full run's settings. Then save this notebook with its outputs (**File → Download → Download .ipynb**) and add it and the files above to `CS336_Colab/` in a pull request. GitHub warns about files over 50 MB and rejects files over 100 MB; `model.pt` stays under both.

In [ ]:
SAMPLE_PROMPTS = (
    "Once upon a time, a little robot found a blue kite.",
    "Lily wanted to bake a cake for her mom, but",
    "Tom and his dog went to the park. Suddenly,",
)
SAMPLE_SETTINGS = dict(max_new_tokens=160, temperature=0.8, top_k=40, seed=42)

def package_results():
    lines = [
        f"# Sample stories · {saved.get('run', run_dir.name)}",
        "",
        f"Iteration {saved['step']:,}"
        + (f" of {saved['total_steps']:,}" if saved.get("total_steps") else "")
        + f" · sampled validation loss {saved['validation_loss']:.4f}"
        + (f" · full validation loss {saved['full_validation_loss']:.4f}" if saved.get("full_validation_loss") is not None else ""),
        "",
        "Sampling: " + ", ".join(f"{key} = {value}" for key, value in SAMPLE_SETTINGS.items()),
        "",
    ]
    for prompt in SAMPLE_PROMPTS:
        story = generate(story_model, story_tokenizer, prompt, **SAMPLE_SETTINGS)
        lines += [f"## {prompt}", "", story.strip(), ""]
        print(story, "\n")

    upload = root / "CS336_Colab_upload"
    upload.mkdir(parents=True, exist_ok=True)
    atomic_write(upload / "samples.md", lambda path: path.write_text("\n".join(lines), encoding="utf-8"))
    for source in (run_dir / "model.pt", run_dir / "metrics.json", run_dir / "learning_curves.png",
                   root / "runs" / "lr_sweep.json", root / "runs" / "lr_sweep.png"):
        if source.exists():
            stage_copy(source, upload / source.name)
    for path in sorted(upload.iterdir()):
        print(f"{path.name:>22}  {path.stat().st_size / 1e6:8.2f} MB")
    print(f"\nUpload folder: {upload}")

if MODE == "sweep":
    print("Package after the full run: set MODE = \"train\" or \"generate\" with the full run's settings.")
else:
    package_results()

### Download the files

Everything is in your Drive under `CS336_TinyStories/CS336_Colab_upload/`. To download a zip of that folder from Colab, set the switch in the next cell to `True` and run it.

In [ ]:
DOWNLOAD_UPLOAD_FOLDER = False
if DOWNLOAD_UPLOAD_FOLDER:
    from google.colab import files
    archive = shutil.make_archive(str(LOCAL_ROOT / "CS336_Colab_upload"), "zip", root / "CS336_Colab_upload")
    files.download(archive)